# Layer 3 — Latenz-Basisstatistiken

Erstes Analyse-Notebook: p50/p95/p99 für Deepgram (STT, `ttft_ms`) und ElevenLabs (TTS, `ttfa_ms`).

**Daten:** `data/layer3/*.jsonl` — Records mit `api="requesty"` werden als deprecated ausgefiltert.

**Ziel:** Zeigen, dass die Layer-3-Timing-Daten statistisch nutzbar sind, unabhängig von `transcript_len=0`.

**Anmerkung zum Feldnamen:** STT misst *time to first token* (`ttft_ms`), TTS misst *time to first audio* (`ttfa_ms`). Semantisch beides die First-Response-Latenz — im Notebook vereinheitlicht als `latency_ms`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/layer3")
print(f"Gefundene Dateien: {len(list(DATA_DIR.glob('*.jsonl')))}")

## Daten laden

In [ ]:
records = []
for path in sorted(DATA_DIR.glob("*.jsonl")):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

df = pd.DataFrame(records)
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
print(f"Total records: {len(df):,}")
print(f"Zeitraum: {df['ts'].min()} → {df['ts'].max()}")
print(f"Unique APIs: {df['api'].dropna().unique().tolist()}")
print(f"Unique Metrics: {df['metric'].dropna().unique().tolist()}")

## Deprecated-Filter: Requesty ausschließen

In [ ]:
before = len(df)
df = df[df["api"] != "requesty"].copy()
print(f"Requesty entfernt: {before - len(df):,} Records")
print(f"Verbleibend: {len(df):,} Records")
print(f"APIs nach Filter: {df['api'].dropna().unique().tolist()}")

## Samples pro Provider/Metric

In [ ]:
samples = df.groupby(["api", "metric"]).size().reset_index(name="n").sort_values(["api", "metric"])
samples

## Basisstatistiken: p50/p95/p99 für `connect_ms` und `latency_ms`

Pro Provider werden nur Einzelmess-Records betrachtet (Summary-Records haben kein `ttft_ms`/`ttfa_ms`).
`latency_ms` = `ttft_ms` (STT) oder `ttfa_ms` (TTS) — die "Zeit bis zum ersten Output".

In [ ]:
def percentiles(series):
    series = series.dropna()
    if len(series) == 0:
        return pd.Series({"n": 0, "p50": np.nan, "p95": np.nan, "p99": np.nan, "mean": np.nan})
    return pd.Series({
        "n": len(series),
        "p50": round(series.quantile(0.50), 1),
        "p95": round(series.quantile(0.95), 1),
        "p99": round(series.quantile(0.99), 1),
        "mean": round(series.mean(), 1),
    })

# Nur Einzelmess-Records (keine Summaries)
events = df[df["metric"].isin(["stt_ttft", "tts_ttfa"])].copy()

# Vereinheitliche: latency_ms = ttft_ms (STT) oder ttfa_ms (TTS)
events["latency_ms"] = events.get("ttft_ms").fillna(events.get("ttfa_ms"))

for col in ["connect_ms", "latency_ms"]:
    if col not in events.columns:
        continue
    print(f"\n=== {col} ===")
    stats = events.groupby(["api", "metric"])[col].apply(percentiles).unstack()
    print(stats.to_string())

## Histogramme der Latenz-Verteilung

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (api, color) in zip(axes, [("deepgram", "steelblue"), ("elevenlabs", "darkorange")]):
    subset = events[(events["api"] == api) & events["latency_ms"].notna()]
    if len(subset) == 0:
        continue
    ax.hist(subset["latency_ms"], bins=60, color=color, edgecolor="white", alpha=0.85)
    p50, p95, p99 = subset["latency_ms"].quantile([0.5, 0.95, 0.99])
    for p, label, style in [(p50, "p50", "-"), (p95, "p95", "--"), (p99, "p99", ":")]:
        ax.axvline(p, color="black", linestyle=style, linewidth=1, label=f"{label}: {p:.0f}ms")
    field = "ttft_ms" if api == "deepgram" else "ttfa_ms"
    ax.set_title(f"{api} — {field} (n={len(subset):,})")
    ax.set_xlabel("latency_ms")
    ax.set_ylabel("count")
    ax.legend()

plt.tight_layout()
plt.show()

## Nächste Schritte

Dieses Notebook zeigt nur die Basisstatistiken. Kommende Analysen (eigene Notebooks):

- **Tageszeit-Trends:** TTFT über `tag` (06h/09h/12h/…) gruppieren
- **Cross-Layer:** Korrelation Layer-1-Ping-RTT ↔ Layer-3-connect_ms
- **E2E Pipeline:** Chain-Records analysieren (sequenzielle Latenz)
- **Layer-2 Protokoll:** PCAP-Analyse der 3-RTT-Overhead